In [6]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.data_selection.data_selector import Data_selector

In [7]:
import pandas as pd
folder_path = os.path.join(project_root, "data", "processed", "integrated.csv")
df_zero = pd.read_csv(folder_path)

In [3]:
folder_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(folder_path)

In [4]:
data_set = {"name":[],"code":[],"gi":[],"g0":[],"g1":[],"g2":[],"g3":[],"g4":[],"g5":[],"n_T":[],"n_G":[],"n_S":[]}
ds = Data_selector(df)
ds_zero = Data_selector(df_zero)
for _,row in df_zero[["name","code"]].drop_duplicates().iterrows():
    name = row["name"]
    code = row["code"]
    data_set["name"].append(name)
    data_set["code"].append(code)
    unit_ds = Data_selector(ds.filter_name_code(name,code))
    
    df_zero_filtered = ds_zero.filter_name_code(name,code)
    ll = len(df_zero_filtered)
    data_set["gi"].append(ll)
    data_set["n_T"].append(int(100*df_zero_filtered['temperature'].isnull().sum()/ll))
    data_set["n_S"].append(int(100*(df_zero_filtered['scadaf'].isnull() | df_zero_filtered['temp_sens'].isnull()).sum()/ll))
    data_set["n_G"].append(int(100*df_zero_filtered['generation'].isnull().sum()/ll))
    
    for goodness in range(0,6):
        gx = f"g{goodness}"
        l = len(unit_ds.select_peaks(goodness))
        data_set[gx].append(int((l/ll)*100))
        

In [8]:
df_data = pd.DataFrame(data_set)

In [9]:
df_data["H1"] = df_data["n_T"] + df_data["g0"]

In [10]:
df_data[df_data["code"].str.startswith("G")]

,name,code,gi,g0,g1,g2,g3,g4,g5,n_T,n_G,n_S,H1
0,پرند,G11,35208,97,86,33,18,13,1,0,1,28,97
1,پرند,G15,12480,99,87,36,17,17,3,0,0,10,99
6,شهدای پاکدشت - دماوند,G14,12480,99,81,35,14,14,11,0,0,37,99
7,شهدای پاکدشت - دماوند,G18,12480,99,93,37,15,15,12,0,0,37,99
8,شهدای پاکدشت - دماوند,G22,12480,99,78,31,12,2,2,0,0,37,99
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,عسلویه,G14,12480,99,88,37,19,19,2,0,0,2,99
98,حافظ,G14,12480,99,85,37,20,20,3,0,0,53,99
99,حافظ,G16,12480,99,85,37,19,19,3,0,0,53,99
100,حافظ,G15,12480,99,87,37,21,21,3,0,0,53,99


In [65]:
def get_null(df, df_name_code):
    infos = []
    for _, row in df_name_code.iterrows():
        name = row["name"]
        code = row["code"]
        unit = Data_selector(df).filter_name_code(name, code)
        g = (unit.isnull().sum() / len(unit)).astype(object)
        g.loc["name"] = name
        g.loc["code"] = code
        infos.append(g)
    df_infos = pd.DataFrame(infos)

    drop_cols = []
    for col in df_infos.columns:
        try:
            m = sum(df_infos[col])
            if m == 0:
                print(col)
                drop_cols.append(col)
        except:
            pass
    df_infos2 = df_infos.drop(columns=drop_cols)
    df_infos2.columns[(df_infos2 == 1).all()]

    return df_infos2